In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, List
from collections import Counter

load_dotenv()

True

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, max_tokens=800)


In [3]:
from schema import CoTResult,ToTResult, FALLBACK
cot_parser = PydanticOutputParser(pydantic_object=CoTResult)


In [4]:
COT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''Expert support email classifier.

Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing + evaluating alternatives → Churn Risk
- Feature requests → Feature Request
- Spam → Spam

Think step by step before classifying.

{format_instructions}'''),
    ('human', 'Subject: {subject}\nBody: {body}')
]).partial(format_instructions=cot_parser.get_format_instructions())

cot_chain = COT_PROMPT | llm | cot_parser

In [5]:
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body': 'Our entire team cannot login. We paid for the annual plan '
            'last week. We have a board demo in 3 hours!'
}

In [7]:
result = cot_chain.invoke(email)

# class CoTResult(BaseModel):
#     category: Literal['Billing','Technical','Feature Request','Churn Risk','Spam','Other']
#     urgency: Literal['High', 'Medium', 'Low']
#     confidence: int = Field(ge=1, le=10)
#     reasoning: str = Field(description='One sentence explanation')
#     cot_steps: List[str] = Field(description='3-5 reasoning steps')

In [8]:
result.cot_steps

['The user mentions they cannot login.',
 'They specify that they paid for the annual plan last week.',
 'The urgency is heightened by the upcoming board demo in 3 hours.',
 'Since the issue is linked to payment, it falls under Billing.']

In [9]:
tot_parser = PydanticOutputParser(pydantic_object=ToTResult)



TOT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''Expert email analyzer using Tree of Thought.

Consider MULTIPLE interpretations before choosing.

Steps:
1. Generate 2-3 different ways to interpret this email
2. Explain reasoning for each
3. Select the best interpretation
4. Give final classification

{format_instructions}'''),
    ('human', 'Subject: {subject}\nBody: {body}')
]).partial(format_instructions=tot_parser.get_format_instructions())


tot_chain = TOT_PROMPT | llm | tot_parser

